In [ ]:
import pandas as pd
import os
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer, util
import torch


In [ ]:
output_path = 'Results_jan2026/'
topic_model = BERTopic.load(output_path + 'bertopic_dpi_6jan2026')
#filtered_topic_model = BERTopic.load(output_path + 'bertopic_dpi_6jan2026') #this one will be classified mannually
topic_info = topic_model.get_topic_info()

In [ ]:
topic_info.Representation

In [ ]:
promoting_keywords_10 = [
    "digital identity",
    "digital payment",
    "development",
    "economic growth",
    "innovation",
    "technology",
    "platform economy",
    "government services",
    "financial inclusion",
    "productivity"
]

safeguarding_keywords_10 = [
    "human rights",
    "privacy",
    "data protection",
    "transparency",
    "accountability",
    "inclusion",
    "trust",
    "freedom",
    "democracy",
    "justice"
]

In [ ]:
promoting_keywords_10 = ["artificial intelligence", "ai", "Big Data", "smart city", "digital ID", "blockchain", "cloud", "cryptocurrencies", "CBDC", "agtech", "fintech",
                         "startups", "innovation hubs", "investors", "banks",
                         "digital transformation", "sustainable transformation", "digital government", "open finance", "investments", "asset management", "automation", "infrastructural development",
                         "public goods", "interoperability","innovation", "economic growth", "added value", "economic security", "trade", "fiscal credibility", "wealth", "patents", "growth", "productivity", "employment", "competitiveness", "technology diffusion"]

safeguarding_keywords_10 = ["cybersecurity", "surveillance economy", "safeguarding devices",
"communities", "diversity group", "social council", "organisation activists", "society organisations",
"social protection", "data protection", "regulation", "citizen engagement", "self determination", "privacy respect", "accountability", "corruption control", "redressal mechanism", "oversight", "online censorship",
"just", "trust", "safe", "universal", "democratic", "civic", "accountable", "agency centred", "biased",
"empowerment", "equality", "accessibility", "justice", "rights", "freedom", "confidence", "transparency", "integrity", "fairness", "social responsibility"]

In [ ]:
# Define keyword lists
promoting_keywords = [
    # Economic and industrial development
    "economic growth", "development", "industrial policy", "innovation", "digital transformation",
    "sustainable development", "competitiveness", "productivity", "efficiency", "employment",
    "infrastructure", "digital infrastructure", "interoperability", "investment", "technology diffusion",
    "automation", "digital economy", "value creation", "wealth", "digital industry", "smart manufacturing",
    "Industry 4.0", "key enabling technologies", "research and development", "education", "human capital",
    "digital skills", "startups", "entrepreneurship", "innovation ecosystem",

    # Technological sovereignty / autonomy
    "technological sovereignty", "strategic autonomy", "digital sovereignty", "data infrastructure",
    "digital ID", "digital identity", "blockchain", "fintech", "agtech", "open finance", "CBDC",
    "digital payments", "open data", "data sharing", "AI", "artificial intelligence", "machine learning",
    "cloud computing", "cyber-physical systems", "smart cities", "platforms", "digital public goods",

    # Governance and cooperation
    "public-private collaboration", "multistakeholder", "partnership", "international cooperation",
    "cross-border", "scalability", "digital government", "Government as a Platform", "digital innovation",
    "knowledge economy", "capacity building", "inclusive growth"
]

safeguarding_keywords = [
    # Social protection and justice
    "social protection", "inclusion", "accessibility", "rights", "justice", "fairness",
    "empowerment", "freedom", "equality", "civic", "citizen engagement", "participation",
    "trust", "confidence", "accountability", "transparency", "integrity",

    # Data, privacy, and cybersecurity
    "data protection", "privacy", "cybersecurity", "digital security", "resilience",
    "privacy-by-design", "data governance", "oversight", "ethical standards", "ethics",
    "algorithmic transparency", "safeguards", "surveillance economy", "regulation",
    "GDPR", "digital rights", "informational self-determination", "data sovereignty",

    # Institutional and normative aspects
    "good governance", "government accountability", "rule of law", "anti-corruption",
    "oversight mechanisms", "redress", "complaints", "ombudsman", "audit", "public trust",
    "inclusion", "universal access", "human-centered design", "digital divide", "vulnerable groups",
    "diversity", "social responsibility", "ethics-by-design", "transparency mechanisms"
]

In [ ]:
# Load the model
model = SentenceTransformer("all-MiniLM-L6-v2")
# Compute keyword centroids (2D tensors) with normalized embeddings
promoting_emb = model.encode(promoting_keywords_10, convert_to_tensor=True, normalize_embeddings=True)
safeguarding_emb = model.encode(safeguarding_keywords_10, convert_to_tensor=True, normalize_embeddings=True)

promoting_vec = promoting_emb.mean(dim=0, keepdim=True)       # shape: (1, dim)
safeguarding_vec = safeguarding_emb.mean(dim=0, keepdim=True) # shape: (1, dim)

def to_text(x) -> str:
    """Ensure the input to the encoder is a single string (topic representations are often lists)."""
    if isinstance(x, list):
        return " ".join(str(t) for t in x)
    return str(x)

# Classification function
def classify_phrase(
    phrase,
    diff_threshold: float = 0.05,
    promoting_threshold: float = 0.3,
    safeguarding_threshold: float = 0.3,
):
    phrase = to_text(phrase)

    # Encode and normalize; enforce 2D shape
    phrase_vec = model.encode(phrase, convert_to_tensor=True, normalize_embeddings=True)
    if phrase_vec.dim() == 1:
        phrase_vec = phrase_vec.unsqueeze(0)  # shape: (1, dim)

    # Cosine similarity returns a (1,1) tensor -> safe to index [0,0]
    promote_score = util.cos_sim(phrase_vec, promoting_vec)[0, 0].item()
    safeguard_score = util.cos_sim(phrase_vec, safeguarding_vec)[0, 0].item()

    if (
        abs(promote_score - safeguard_score) < diff_threshold
        or (promote_score < promoting_threshold and safeguard_score < safeguarding_threshold)
    ):
        classification = "neutral"
    elif promote_score > safeguard_score:
        classification = "promoting"
    else:
        classification = "safeguarding"

    return classification, promote_score, safeguard_score


In [ ]:
topic_info["Representation_text"] = topic_info["Representation"].apply(to_text)

topic_info[["DPI_Classification", "Promoting_Score", "Safeguarding_Score"]] = (
    topic_info["Representation_text"]
    .apply(lambda x: pd.Series(classify_phrase(
        x, diff_threshold=0.05, promoting_threshold=0.4, safeguarding_threshold=0.3
    )))
)


In [ ]:
topic_info.DPI_Classification.value_counts()

In [ ]:
topic_info['Diff_Score'] = topic_info.Promoting_Score - topic_info.Safeguarding_Score

Classification of topics based on the presence of keywords

In [ ]:
promoting_dpi_keywords = {
    "technologies": ["artificial intelligence", "ai", "Big Data", "smart city", "digital ID", "blockchain", "cloud", "cryptocurrencies", "CBDC", "agtech", "fintech"],
    "actors": ["startups", "startup", "innovation hubs", "innovation hub", "investors", "investor", "bank", "banks"],
    "processes": ["digital transformation", "sustainable transformation", "digital government", "open finance", "investments", "investment", "asset management", "automation", "infrastructural development"],
    "attributes": ["public goods", "public good", "interoperability"],
    "results": ["innovation", "economic growth", "added value", "economic security", "trade", "fiscal credibility", "wealth", "patents", "patent", "growth", "productivity", "employment", "competitiveness", "technology diffusion"]
}

safeguarding_dpi_keywords = {
    "technologies": ["cybersecurity", "surveillance economy", "safeguarding devices", "safeguarding device"],
    "actors": ["communities", "community", "diversity group", "social council", "organisation activists", "society organisations", "citizens"],
    "processes": ["social protection", "data protection", "regulation", "citizen engagement", "self determination", "privacy respect", "accountability", "corruption control", "redressal mechanism", "oversight", "online censorship"],
    "attributes": ["just", "trust", "safe", "universal", "democratic", "civic", "accountable", "agency centred", "biased"],
    "results": ["empowerment", "equality", "accessibility", "justice", "rights", "freedom", "confidence", "transparency", "integrity", "fairness", "social responsibility"]
}

# Complete list of keywords for Promoting DPI
promoting_dpi_all_keywords = [
    keyword for sublist in promoting_dpi_keywords.values() for keyword in sublist
]

# Complete list of keywords for Safeguarding DPI
safeguarding_dpi_all_keywords = [
    keyword for sublist in safeguarding_dpi_keywords.values() for keyword in sublist
]

# Variables with sublists for each part of Promoting DPI
promoting_technologies = promoting_dpi_keywords["technologies"]
promoting_actors = promoting_dpi_keywords["actors"]
promoting_processes = promoting_dpi_keywords["processes"]
promoting_attributes = promoting_dpi_keywords["attributes"]
promoting_results = promoting_dpi_keywords["results"]

# Variables with sublists for each part of Safeguarding DPI
safeguarding_technologies = safeguarding_dpi_keywords["technologies"]
safeguarding_actors = safeguarding_dpi_keywords["actors"]
safeguarding_processes = safeguarding_dpi_keywords["processes"]
safeguarding_attributes = safeguarding_dpi_keywords["attributes"]
safeguarding_results = safeguarding_dpi_keywords["results"]

# Print the complete lists for verification
print("Promoting DPI Keywords:", promoting_dpi_all_keywords)
print("Safeguarding DPI Keywords:", safeguarding_dpi_all_keywords)

# Function to check if any keyword from the seed list is present in the Representation list (case-insensitive)
def contains_keyword(representation, seed_list):
    rep_str = str(representation).lower()
    return int(any(keyword.lower() in rep_str for keyword in seed_list))

In [ ]:
topic_info['Promoting'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_dpi_all_keywords))
topic_info['Safeguarding'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_dpi_all_keywords))
topic_info['Promoting_technologies'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_technologies))
topic_info['Promoting_actors'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_actors))
topic_info['Promoting_processes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_processes))
topic_info['Promoting_attributes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_attributes))
topic_info['Promoting_results'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, promoting_results))
topic_info['Safeguarding_technologies'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_technologies))
topic_info['Safeguarding_actors'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_actors))
topic_info['Safeguarding_processes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_processes))
topic_info['Safeguarding_attributes'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_attributes))
topic_info['Safeguarding_results'] = topic_info['Representation'].apply(lambda x: contains_keyword(x, safeguarding_results))

In [ ]:
topic_info.describe()

In [ ]:
topic_info.to_excel(output_path + 'bertopic_dpi_autoclass_9jan2026.xlsx', index=False)